# 04 - Stress testing and expected credit loss


Illustrative borrower-level stresses are applied to key PD drivers and LGD. The portfolio is re-scored and translated into expected credit loss using **ECL = PD x EAD x LGD**. These are sensitivity scenarios, not regulatory forecasts.


In [1]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import generate_portfolio, FEATURES, TARGET
from src.model import build_logistic_model
from src.stress import apply_stress, stressed_lgd, expected_credit_loss
df = generate_portfolio()
train = df[df.observation_year <= 2024]
test = df[df.observation_year == 2025]
model = build_logistic_model().fit(train[FEATURES], train[TARGET])
rows = []
for scenario in ['baseline', 'mild', 'severe']:
    stressed = apply_stress(test[FEATURES], scenario)
    pd_values = model.predict_proba(stressed[FEATURES])[:, 1]
    lgd = stressed_lgd(test['loss_given_default'], scenario)
    ecl = expected_credit_loss(pd_values, test['exposure_at_default'], lgd)
    rows.append({'scenario': scenario, 'mean_pd': pd_values.mean(), 'mean_lgd': lgd.mean(), 'expected_loss': ecl.sum()})
pd.DataFrame(rows).round(4)


,scenario,mean_pd,mean_lgd,expected_loss
0,baseline,0.1085,0.4148,1.281832e+06
1,mild,0.1340,0.4648,1.767941e+06
2,severe,0.2029,0.5148,2.944529e+06
